
# Agente 1 del TFM: lenguaje natural → `yfinance` → CSV

Este notebook implementa una **primera versión funcional** del primer agente de tu arquitectura:

**mensaje del usuario** → **parser en lenguaje natural** → **resolución del activo a ticker** → **descarga con `yfinance`** → **exportación a CSV**

## Qué resuelve

Acepta peticiones como estas:

- `Cuánto ha crecido Nvidia en 5 años`
- `Descárgame el histórico del S&P 500 desde 2020`
- `Quiero el oro en 1 semana a 1h`
- `Compara Nvidia y AMD en 2 años`
- `Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31`

## Enfoque usado

La solución es **híbrida**:

1. **Reglas de parsing** para extraer activos, rango temporal e intervalo.
2. **Diccionario corto de alias** para casos muy frecuentes en español.
3. **Resolución dinámica con `yfinance.Search`** para activos no cubiertos por alias.
4. **Caché local** (`ticker_cache.json`) para reutilizar resoluciones ya encontradas.
5. **Descarga final con `yf.download(...)`** y guardado del resultado en CSV.

## Notas de diseño

- El notebook usa `start` y `end` para acercarse al patrón que ya tienes en tu script.
- `yfinance` trata `end` como **fecha exclusiva**, así que cuando el usuario pide `hasta 2024-12-31`, internamente se usa `2025-01-01` para incluir el 31 de diciembre.
- Si el usuario pide un **intervalo intradía** (`1m`, `5m`, `1h`, etc.) para un rango mayor de 60 días, el notebook cambia automáticamente el intervalo a `1d`, porque Yahoo Finance no ofrece histórico intradía más allá de ese límite.


In [1]:
%pip install -q yfinance pandas python-dateutil

Note: you may need to restart the kernel to use updated packages.



## 1. Imports y configuración base


In [2]:

import re
import json
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple
from datetime import datetime

import pandas as pd
from dateutil.relativedelta import relativedelta
import yfinance as yf



## 2. Diccionario de alias y utilidades generales

Aquí dejamos un conjunto **pequeño y mantenible** de alias comunes.  
No intentamos meter miles de tickers a mano: para eso usaremos la búsqueda dinámica de `yfinance`.


In [3]:

VALID_INTERVALS = {
    "1m", "2m", "5m", "15m", "30m", "60m", "90m",
    "1h", "1d", "5d", "1wk", "1mo", "3mo"
}

INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}


def normalize_text(text: str) -> str:
    text = text.strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


ALIASES_RAW = {
    # Acciones frecuentes
    "nvidia": "NVDA",
    "amd": "AMD",
    "apple": "AAPL",
    "microsoft": "MSFT",
    "tesla": "TSLA",
    "amazon": "AMZN",
    "google": "GOOG",
    "alphabet": "GOOG",
    "meta": "META",
    "netflix": "NFLX",

    # Índices
    "sp500": "^GSPC",
    "s&p 500": "^GSPC",
    "s p 500": "^GSPC",
    "sandp 500": "^GSPC",
    "nasdaq": "^IXIC",
    "nasdaq 100": "^NDX",
    "dow jones": "^DJI",
    "ibex 35": "^IBEX",
    "euro stoxx 50": "^STOXX50E",
    "dax": "^GDAXI",
    "cac 40": "^FCHI",
    "nikkei 225": "^N225",

    # Cripto
    "bitcoin": "BTC-USD",
    "btc": "BTC-USD",
    "ethereum": "ETH-USD",
    "eth": "ETH-USD",

    # Materias primas / futuros
    "oro": "GC=F",
    "gold": "GC=F",
    "plata": "SI=F",
    "silver": "SI=F",
    "petroleo": "CL=F",
    "petróleo": "CL=F",
    "crudo": "CL=F",
    "brent": "BZ=F",
    "gas natural": "NG=F",
    "cobre": "HG=F",

    # Divisas
    "eurusd": "EURUSD=X",
    "eur/usd": "EURUSD=X",
    "usd/jpy": "JPY=X",
    "gbp/usd": "GBPUSD=X",
}

ALIASES = {normalize_text(k): v for k, v in ALIASES_RAW.items()}


STOP_PHRASES = [normalize_text(x) for x in [
    "cuanto ha crecido", "cuánto ha crecido",
    "descargame el historico del", "descárgame el histórico del",
    "descargame el histórico del", "descargame los datos de", "descárgame los datos de",
    "descargame", "descárgame",
    "quiero el", "quiero la", "quiero los", "quiero las", "quiero",
    "datos de",
    "historico del", "histórico del", "historico de", "histórico de",
    "compara", "comparame", "compárame", "comparar",
    "cotizacion de", "cotización de",
    "precio de", "precios de",
]]


UNIT_MAP = {
    "ano": "years", "anos": "years", "año": "years", "años": "years",
    "year": "years", "years": "years",
    "mes": "months", "meses": "months",
    "month": "months", "months": "months",
    "semana": "weeks", "semanas": "weeks",
    "week": "weeks", "weeks": "weeks",
    "dia": "days", "dias": "days", "día": "days", "días": "days",
    "day": "days", "days": "days",
    "hora": "hours", "horas": "hours",
    "hour": "hours", "hours": "hours",
}


def looks_like_ticker(text: str) -> bool:
    return bool(re.fullmatch(r"[\^A-Z0-9][A-Z0-9=\-\.]{0,14}", text.strip().upper()))


def today_floor() -> pd.Timestamp:
    return pd.Timestamp.utcnow().tz_localize(None).normalize()


def to_yyyy_mm_dd(ts: pd.Timestamp) -> str:
    return pd.Timestamp(ts).strftime("%Y-%m-%d")


def inclusive_to_exclusive_end(end_date: str) -> str:
    return (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def slugify(text: str) -> str:
    text = strip_accents(text.lower())
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")[:80] or "consulta"



## 3. Parser de lenguaje natural

Este bloque convierte un mensaje como:

> `Quiero el oro en 1 semana a 1h`

en una estructura como:

```python
{
    "assets": ["oro"],
    "start": "2026-03-05",
    "end": "2026-03-13",
    "interval": "1h"
}
```

La idea no es “entender el mundo”, sino **extraer bien los parámetros operativos que necesita `yfinance`**.


In [4]:

UNIT_RE = (
    r"(ano|anos|año|años|mes|meses|semana|semanas|dia|dias|día|días|"
    r"hora|horas|year|years|month|months|week|weeks|day|days|hour|hours)"
)


def apply_relative_range(qty: int, unit_text: str) -> Tuple[str, str]:
    unit_text = normalize_text(unit_text)
    unit_key = UNIT_MAP[unit_text]
    now = today_floor()

    if unit_key == "years":
        start = now - relativedelta(years=qty)
    elif unit_key == "months":
        start = now - relativedelta(months=qty)
    elif unit_key == "weeks":
        start = now - relativedelta(weeks=qty)
    elif unit_key == "days":
        start = now - relativedelta(days=qty)
    elif unit_key == "hours":
        start = (pd.Timestamp.utcnow().tz_localize(None) - relativedelta(hours=qty)).floor("min")
    else:
        start = now - relativedelta(years=1)

    end = now + pd.Timedelta(days=1)
    return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end)


def extract_interval(text: str) -> Tuple[str, str]:
    interval = "1d"
    patterns = [
        r"(?:a|cada|intervalo(?: de)?)\s*(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1mo|3mo)\b",
        r"\b(1m|2m|5m|15m|30m|60m|90m|1h|1d|5d|1wk|1mo|3mo)\b",
    ]
    cleaned = text

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            interval = match.group(1)
            cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)
            break

    return interval, re.sub(r"\s+", " ", cleaned).strip()


def extract_date_range(text: str) -> Tuple[str, str, str, List[str]]:
    notes: List[str] = []
    cleaned = text

    # Caso: desde YYYY-MM-DD hasta YYYY-MM-DD
    between = re.search(
        r"desde\s+(\d{4}-\d{2}-\d{2}|\d{4})\s+hasta\s+(\d{4}-\d{2}-\d{2}|\d{4})",
        text,
        flags=re.IGNORECASE,
    )
    if between:
        raw_start, raw_end = between.group(1), between.group(2)
        start = f"{raw_start}-01-01" if re.fullmatch(r"\d{4}", raw_start) else raw_start
        end_inclusive = f"{raw_end}-12-31" if re.fullmatch(r"\d{4}", raw_end) else raw_end
        cleaned = cleaned.replace(between.group(0), " ")
        return start, inclusive_to_exclusive_end(end_inclusive), re.sub(r"\s+", " ", cleaned).strip(), notes

    # Caso: desde 2020 / desde 2024-01-01
    since = re.search(r"desde\s+(\d{4}-\d{2}-\d{2}|\d{4})", text, flags=re.IGNORECASE)
    if since:
        raw_start = since.group(1)
        start = f"{raw_start}-01-01" if re.fullmatch(r"\d{4}", raw_start) else raw_start
        end = to_yyyy_mm_dd(today_floor() + pd.Timedelta(days=1))
        cleaned = cleaned.replace(since.group(0), " ")
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    # Caso: en 5 años / últimos 2 meses / de 10 días
    numeric_rel = re.search(
        rf"(?:en|ultimos?|últimos?|ultimas?|últimas?|de)\s+(\d+)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if numeric_rel:
        qty = int(numeric_rel.group(1))
        unit = numeric_rel.group(2)
        start, end = apply_relative_range(qty, unit)
        cleaned = cleaned.replace(numeric_rel.group(0), " ")
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    # Caso: en una semana / último mes / última semana
    single_rel = re.search(
        rf"(?:en|ultimo|último|ultima|última)\s+(un|una)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if single_rel:
        unit = single_rel.group(2)
        start, end = apply_relative_range(1, unit)
        cleaned = cleaned.replace(single_rel.group(0), " ")
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    last_rel = re.search(
        rf"(?:ultimo|último|ultima|última)\s+{UNIT_RE}\b",
        text,
        flags=re.IGNORECASE,
    )
    if last_rel:
        unit = last_rel.group(1)
        start, end = apply_relative_range(1, unit)
        cleaned = cleaned.replace(last_rel.group(0), " ")
        return start, end, re.sub(r"\s+", " ", cleaned).strip(), notes

    # Caso: YTD / este año
    if re.search(r"\bytd\b|este ano|este año", text, flags=re.IGNORECASE):
        now = today_floor()
        start = pd.Timestamp(year=now.year, month=1, day=1)
        end = now + pd.Timedelta(days=1)
        cleaned = re.sub(r"\bytd\b|este ano|este año", " ", cleaned, flags=re.IGNORECASE)
        return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end), re.sub(r"\s+", " ", cleaned).strip(), notes

    # Fallback
    now = today_floor()
    start = now - relativedelta(years=1)
    end = now + pd.Timedelta(days=1)
    notes.append("No se detectó rango temporal; se usa por defecto el último año.")
    return to_yyyy_mm_dd(start), to_yyyy_mm_dd(end), re.sub(r"\s+", " ", cleaned).strip(), notes


def clean_asset_chunk(text: str) -> str:
    cleaned = normalize_text(text)
    for phrase in sorted(STOP_PHRASES, key=len, reverse=True):
        cleaned = re.sub(rf"\b{re.escape(phrase)}\b", " ", cleaned)
    cleaned = re.sub(r"\bdel\b|\bde\b|\bla\b|\bel\b|\blos\b|\blas\b", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,;:.")
    return cleaned


def extract_assets(text: str) -> List[str]:
    cleaned = clean_asset_chunk(text)
    if not cleaned:
        return []

    parts = re.split(r"\s*(?:,| y | e | vs | versus | contra )\s*", cleaned)
    assets = [part.strip(" ,;:.") for part in parts if part.strip(" ,;:.")]
    return assets


def adjust_interval_for_range(start: str, end: str, interval: str) -> Tuple[str, List[str]]:
    notes: List[str] = []
    delta_days = (pd.Timestamp(end) - pd.Timestamp(start)).days

    if interval in INTRADAY_INTERVALS and delta_days > 60:
        notes.append(
            f"El intervalo solicitado ({interval}) es intradía y el rango supera 60 días. "
            "Se cambia automáticamente a 1d para respetar la limitación de Yahoo Finance."
        )
        return "1d", notes

    return interval, notes


def parse_user_request(user_text: str) -> Dict[str, Any]:
    interval, text_wo_interval = extract_interval(user_text)
    start, end, text_wo_dates, notes = extract_date_range(text_wo_interval)
    assets = extract_assets(text_wo_dates)

    if not assets:
        raise ValueError("No he podido detectar el activo solicitado en el mensaje.")

    interval, interval_notes = adjust_interval_for_range(start, end, interval)
    notes.extend(interval_notes)

    return {
        "original_query": user_text,
        "assets": assets,
        "start": start,
        "end": end,
        "interval": interval,
        "notes": notes,
    }



### Prueba rápida del parser


In [5]:

examples = [
    "Cuánto ha crecido Nvidia en 5 años",
    "Descárgame el histórico del S&P 500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara Nvidia y AMD en 2 años",
    "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
]

for ex in examples:
    print("=" * 90)
    print(ex)
    print(json.dumps(parse_user_request(ex), ensure_ascii=False, indent=2))


Cuánto ha crecido Nvidia en 5 años
{
  "original_query": "Cuánto ha crecido Nvidia en 5 años",
  "assets": [
    "nvidia"
  ],
  "start": "2021-03-12",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Descárgame el histórico del S&P 500 desde 2020
{
  "original_query": "Descárgame el histórico del S&P 500 desde 2020",
  "assets": [
    "s&p 500"
  ],
  "start": "2020-01-01",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Quiero el oro en 1 semana a 1h
{
  "original_query": "Quiero el oro en 1 semana a 1h",
  "assets": [
    "oro"
  ],
  "start": "2026-03-05",
  "end": "2026-03-13",
  "interval": "1h",
  "notes": []
}
Compara Nvidia y AMD en 2 años
{
  "original_query": "Compara Nvidia y AMD en 2 años",
  "assets": [
    "nvidia",
    "amd"
  ],
  "start": "2024-03-12",
  "end": "2026-03-13",
  "interval": "1d",
  "notes": []
}
Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31
{
  "original_query": "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
  "assets": 

C:\Users\cruizoya\AppData\Local\Temp\ipykernel_24612\1516633543.py:111: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  return pd.Timestamp.utcnow().tz_localize(None).normalize()



## 4. Resolución híbrida activo → ticker

La estrategia es esta:

1. Si el usuario ya escribió un ticker (`NVDA`, `BTC-USD`, `^GSPC`), se usa directamente.
2. Si existe un alias local conocido (`oro`, `bitcoin`, `S&P 500`), se usa ese.
3. Si no, se hace una búsqueda dinámica con `yfinance.Search(...)`.
4. El resultado se guarda en `ticker_cache.json`.


In [6]:

def score_candidate(query: str, candidate: Dict[str, Any]) -> float:
    q = normalize_text(query)
    symbol = normalize_text(str(candidate.get("symbol", "")))
    shortname = normalize_text(str(candidate.get("shortname", "")))
    longname = normalize_text(str(candidate.get("longname", "")))
    quote_type = normalize_text(str(candidate.get("quoteType", "")))
    exchange = normalize_text(str(candidate.get("exchange", "")))

    score = 0.0

    if symbol == q:
        score += 100
    if q == shortname:
        score += 60
    if q == longname:
        score += 60
    if q in shortname:
        score += 30
    if q in longname:
        score += 30
    if q.replace(" ", "") == symbol.replace(" ", ""):
        score += 40
    if quote_type in {"equity", "etf", "index", "cryptocurrency", "currency", "future", "mutualfund"}:
        score += 10
    if exchange in {"nms", "nyq", "nas", "nyse", "nasdaq", "ccc"}:
        score += 5

    return score


def search_yfinance_candidates(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    try:
        search = yf.Search(
            query=query,
            max_results=max_results,
            enable_fuzzy_query=True,
            news_count=0,
        )
        return getattr(search, "quotes", []) or []
    except Exception:
        return []


def resolve_asset_to_ticker(asset: str, cache_path: str = "ticker_cache.json") -> Dict[str, Any]:
    cache_file = Path(cache_path)

    if cache_file.exists():
        try:
            cache = json.loads(cache_file.read_text(encoding="utf-8"))
        except Exception:
            cache = {}
    else:
        cache = {}

    key = normalize_text(asset)

    # 1) Caché
    if key in cache:
        return {
            "query": asset,
            "ticker": cache[key]["ticker"],
            "source": "cache",
            "match": cache[key],
        }

    # 2) Ticker directo
    if looks_like_ticker(asset):
        result = {
            "query": asset,
            "ticker": asset.strip().upper(),
            "source": "direct_ticker",
            "match": {"symbol": asset.strip().upper()},
        }
        cache[key] = {"ticker": result["ticker"], "source": result["source"]}
        cache_file.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
        return result

    # 3) Alias local
    if key in ALIASES:
        result = {
            "query": asset,
            "ticker": ALIASES[key],
            "source": "alias",
            "match": {"symbol": ALIASES[key]},
        }
        cache[key] = {"ticker": result["ticker"], "source": result["source"]}
        cache_file.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
        return result

    # 4) Búsqueda dinámica en Yahoo Finance / yfinance
    candidates = search_yfinance_candidates(asset)
    if not candidates:
        raise ValueError(f"No se pudo resolver el activo '{asset}' en Yahoo Finance / yfinance.")

    ranked = sorted(candidates, key=lambda c: score_candidate(asset, c), reverse=True)
    best = ranked[0]
    ticker = best.get("symbol")

    if not ticker:
        raise ValueError(f"No se encontró ticker válido para '{asset}'.")

    result = {
        "query": asset,
        "ticker": ticker,
        "source": "yfinance_search",
        "match": best,
        "alternatives": ranked[:5],
    }

    cache[key] = {"ticker": ticker, "source": result["source"]}
    cache_file.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
    return result



## 5. Construcción del plan de descarga y exportación a CSV

Aquí se monta exactamente la llamada estilo:

```python
raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    interval=INTERVAL,
    group_by="ticker",
    auto_adjust=False,
    threads=True,
    progress=False
)
```

Además:
- se exporta el `raw` a CSV;
- se genera un archivo `.metadata.json` con trazabilidad.


In [7]:

def build_download_params(
    parsed: Dict[str, Any],
    resolved_assets: List[Dict[str, Any]],
    auto_adjust: bool = False,
) -> Dict[str, Any]:
    return {
        "tickers": [item["ticker"] for item in resolved_assets],
        "start": parsed["start"],
        "end": parsed["end"],
        "interval": parsed["interval"],
        "group_by": "ticker",
        "auto_adjust": auto_adjust,
        "threads": True,
        "progress": False,
    }


def run_user_request(
    user_text: str,
    export_dir: str = "exports",
    auto_adjust: bool = False,
    cache_path: str = "ticker_cache.json",
) -> Dict[str, Any]:
    parsed = parse_user_request(user_text)
    resolved_assets = [resolve_asset_to_ticker(asset, cache_path=cache_path) for asset in parsed["assets"]]
    params = build_download_params(parsed, resolved_assets, auto_adjust=auto_adjust)

    raw = yf.download(
        tickers=params["tickers"],
        start=params["start"],
        end=params["end"],
        interval=params["interval"],
        group_by=params["group_by"],
        auto_adjust=params["auto_adjust"],
        threads=params["threads"],
        progress=params["progress"],
    )

    if raw is None or raw.empty:
        raise ValueError("La descarga no devolvió datos. Revisa el ticker, el rango o el intervalo.")

    export_path = Path(export_dir)
    export_path.mkdir(parents=True, exist_ok=True)

    filename_base = slugify(user_text) + "_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = export_path / f"{filename_base}.csv"
    metadata_path = export_path / f"{filename_base}.metadata.json"

    raw.to_csv(csv_path)

    metadata = {
        "user_text": user_text,
        "parsed_request": parsed,
        "resolved_assets": resolved_assets,
        "download_params": params,
        "csv_path": str(csv_path),
        "row_count": int(len(raw)),
        "column_count": int(len(raw.columns)),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    return {
        "raw": raw,
        "csv_path": str(csv_path),
        "metadata_path": str(metadata_path),
        "metadata": metadata,
    }



## 6. Uso real: una petición del usuario

Cambia el texto de `user_text` y ejecuta la celda.  
Si todo va bien, se descargará el histórico, se guardará un CSV y se mostrará una vista previa.


In [8]:

user_text = "Muestrame cuanto a evolucionado Apple en 5 años"

result = run_user_request(
    user_text=user_text,
    export_dir="exports",
    auto_adjust=False,   # para replicar tu patrón actual
)

print("CSV generado:", result["csv_path"])
print("Metadata:", result["metadata_path"])
print()
print(json.dumps(result["metadata"], ensure_ascii=False, indent=2))
print()
result["raw"].head()


C:\Users\cruizoya\AppData\Local\Temp\ipykernel_24612\1516633543.py:111: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  return pd.Timestamp.utcnow().tz_localize(None).normalize()


ValueError: No se pudo resolver el activo 'muestrame cuanto a evolucionado apple' en Yahoo Finance / yfinance.


## 7. Procesar varias peticiones seguidas

Esta celda es útil para probar varios ejemplos del TFM.


In [9]:

requests_batch = [
    "Cuánto ha crecido Nvidia en 5 años",
    "Descárgame el histórico del S&P 500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara Nvidia y AMD en 2 años",
    "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
]

batch_results = []

for req in requests_batch:
    try:
        output = run_user_request(req, export_dir="exports", auto_adjust=False)
        batch_results.append({
            "query": req,
            "csv_path": output["csv_path"],
            "tickers": output["metadata"]["download_params"]["tickers"],
            "rows": output["metadata"]["row_count"],
        })
    except Exception as e:
        batch_results.append({
            "query": req,
            "error": str(e),
        })

pd.DataFrame(batch_results)


C:\Users\cruizoya\AppData\Local\Temp\ipykernel_24612\1516633543.py:111: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  return pd.Timestamp.utcnow().tz_localize(None).normalize()
Failed to get ticker 'NVIDIA' reason: Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['NVIDIA']: CertificateVerifyError('Failed to perform, curl: (60) SSL certificate problem: unable to get local issuer certificate. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
C:\Users\cruizoya\AppData\Local\Temp\ipykernel_24612\1516633543.py:111: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  return pd.Timestamp.utcnow().tz_localize(None).normalize()
Failed to get ticker '^GSPC' reason: Failed to perform, 

,query,error
0,Cuánto ha crecido Nvidia en 5 años,La descarga no devolvió datos. Revisa el ticke...
1,Descárgame el histórico del S&P 500 desde 2020,La descarga no devolvió datos. Revisa el ticke...
2,Quiero el oro en 1 semana a 1h,La descarga no devolvió datos. Revisa el ticke...
3,Compara Nvidia y AMD en 2 años,La descarga no devolvió datos. Revisa el ticke...
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,La descarga no devolvió datos. Revisa el ticke...



## 8. Ideas de mejora para la siguiente versión

Este notebook ya te deja una base sólida para el **Agente 1** del TFM, pero puedes mejorarlo fácilmente:

- añadir más alias en español;
- manejar mejor ambigüedades (`oro` futuro vs ETF, `nasdaq` índice vs ETF);
- incorporar una capa LLM para consultas más complejas;
- añadir validaciones de mercado o país;
- devolver también un objeto estructurado tipo `FinanceQuery`;
- conectar este notebook con el **Agente 2 Analista** para que procese el CSV generado.

## Resumen operativo

La función principal que debes reutilizar es:

```python
run_user_request("Compara Nvidia y AMD en 2 años")
```

y te devolverá:
- `raw`: el `DataFrame` descargado;
- `csv_path`: la ruta del CSV exportado;
- `metadata_path`: la ruta del JSON con trazabilidad;
- `metadata`: el diccionario completo de ejecución.
